# Emotion Classification: Fine-tuning BERT on GoEmotions & Transfer to Customer Support Tweets

**Module:** MSIN0221 — Natural Language Processing  
**Institution:** University College London (UCL)  
**Runtime:** Google Colab — T4 GPU recommended

---

## Project Overview

This notebook fine-tunes `bert-base-uncased` on the **GoEmotions** dataset (27-class emotion classification derived from Reddit comments) and then evaluates how well the learned representations **transfer** to customer support tweets from the **TWCS** (Twitter Customer Support) corpus.

---

## How to Run

1. **Open in Google Colab** — click `Open in Colab` or upload this `.ipynb` directly.
2. **Set runtime to T4 GPU** — `Runtime → Change runtime type → T4 GPU`.
3. **Upload data** — place `twcs.csv` at `./Data/twcs/twcs.csv` (or mount Google Drive and update `TWITTER_PATH` in Section 2).
4. **Run all cells in order** — `Runtime → Run all` or execute section by section.
5. Model checkpoints are saved to `./outputs/` by default.

---

## Folder Structure

```
NLP-Sentimental-Analysis/
├── project.ipynb          # Main notebook (this file)
├── requirements.txt       # Auto-generated by Section 1
├── Data/
│   └── twcs/
│       └── twcs.csv       # Twitter Customer Support corpus
├── outputs/               # Model checkpoints & logs (created at runtime)
└── figures/               # Saved plots (created at runtime)
```

---

> **Note:** GoEmotions is loaded automatically via the Hugging Face `datasets` library. No manual download is required.

---
# Section 1 — Setup & Installation

This section handles all environment configuration:
- Installs required Python packages
- Imports all libraries used throughout the notebook
- Seeds all random number generators for full reproducibility
- Verifies GPU availability and prints device information
- Writes a `requirements.txt` snapshot of the installed package versions

### 1.1 — Package Installation

Install all project dependencies. The `-q` flag suppresses verbose pip output to keep the log clean.  
If running locally (outside Colab), you can skip this cell and install via `pip install -r requirements.txt` instead.

In [ ]:
# Install all required packages
# transformers  — BERT model, tokeniser, Trainer API
# datasets      — Hugging Face dataset hub (GoEmotions)
# torch         — deep learning backend
# scikit-learn  — evaluation metrics (F1, classification report)
# pandas        — tabular data manipulation
# numpy         — numerical operations
# matplotlib    — base plotting
# seaborn       — statistical visualisation
# accelerate    — required by Hugging Face Trainer for GPU training

!pip install -q \
    transformers \
    datasets \
    torch \
    scikit-learn \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    accelerate

print("All packages installed successfully.")

### 1.2 — Library Imports

Centralising all imports here makes dependencies explicit and prevents hidden import errors later in the notebook.

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import random
import warnings

# ── Numerical / data ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Deep learning ────────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader

# ── Hugging Face — datasets ───────────────────────────────────────────────────
from datasets import load_dataset, DatasetDict

# ── Hugging Face — transformers ───────────────────────────────────────────────
from transformers import (
    BertTokenizerFast,          # Fast tokeniser for bert-base-uncased
    BertForSequenceClassification,  # BERT + classification head
    TrainingArguments,          # Hyperparameter configuration for Trainer
    Trainer,                    # High-level training loop
    EarlyStoppingCallback,      # Stop training when val loss plateaus
    set_seed as hf_set_seed,    # Hugging Face's own seed setter
)

# ── Scikit-learn — evaluation ─────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,      # Per-class precision / recall / F1
    confusion_matrix,           # Confusion matrix for error analysis
    f1_score,                   # Macro / weighted F1
    accuracy_score,             # Overall accuracy
)
from sklearn.preprocessing import LabelEncoder

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

print("All libraries imported successfully.")

### 1.3 — Reproducibility: Random Seed Configuration

Setting a fixed seed across **all** random number generators (Python `random`, NumPy, PyTorch CPU, PyTorch CUDA, and Hugging Face) ensures that results are fully reproducible across runs, which is a requirement for academic reporting.  

`CUBLAS_WORKSPACE_CONFIG` enforces deterministic cuBLAS operations; without it some CUDA ops remain non-deterministic even with a fixed seed.

In [ ]:
SEED = 42  # Project-wide random seed — change here to replicate with a different seed

def set_all_seeds(seed: int) -> None:
    """Set random seeds across every relevant library for full reproducibility."""
    random.seed(seed)                          # Python built-in random
    np.random.seed(seed)                       # NumPy
    torch.manual_seed(seed)                    # PyTorch CPU
    torch.cuda.manual_seed(seed)               # PyTorch CUDA (single GPU)
    torch.cuda.manual_seed_all(seed)           # PyTorch CUDA (multi-GPU)
    hf_set_seed(seed)                          # Hugging Face Trainer seed
    # Make cuBLAS operations deterministic (small performance cost)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

set_all_seeds(SEED)
print(f"Random seed set to {SEED} across: random, numpy, torch (CPU+CUDA), huggingface.")

### 1.4 — GPU Availability Check

Verify that a CUDA-capable GPU is detected. Training BERT on GoEmotions (~211k examples) on CPU is not feasible in a reasonable time — a T4 GPU reduces fine-tuning from hours to ~20 minutes.  

If no GPU is found, the notebook will still run on CPU for data exploration but **training will be slow**.

In [ ]:
# Determine the compute device to be used throughout training
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)          # Human-readable GPU name
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9  # GB
    print(f"GPU detected  : {gpu_name}")
    print(f"VRAM          : {gpu_mem:.1f} GB")
    print(f"CUDA version  : {torch.version.cuda}")
    print(f"PyTorch build : {torch.__version__}")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: No GPU detected. Training will run on CPU (very slow).")
    print("In Colab: Runtime → Change runtime type → T4 GPU, then re-run.")

print(f"\nActive device : {DEVICE}")

### 1.5 — Generate `requirements.txt`

Capture exact installed versions of all project dependencies and write them to `requirements.txt`. This snapshot makes the environment reproducible for collaborators and for the project submission.

In [ ]:
import importlib

# Map display name → importable module name
PACKAGES = {
    "transformers" : "transformers",
    "datasets"     : "datasets",
    "torch"        : "torch",
    "scikit-learn" : "sklearn",
    "pandas"       : "pandas",
    "numpy"        : "numpy",
    "matplotlib"   : "matplotlib",
    "seaborn"      : "seaborn",
    "accelerate"   : "accelerate",
}

lines = []
for pkg_name, import_name in PACKAGES.items():
    try:
        mod     = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        lines.append(f"{pkg_name}=={version}")
        print(f"  {pkg_name:<15} {version}")
    except ImportError:
        lines.append(f"{pkg_name}  # NOT FOUND")
        print(f"  {pkg_name:<15} NOT FOUND")

# Write to disk
req_path = "requirements.txt"
with open(req_path, "w") as f:
    f.write("# Auto-generated by project.ipynb — Section 1\n")
    f.write("\n".join(lines) + "\n")

print(f"\nrequirements.txt written to: {os.path.abspath(req_path)}")

### Section 1 — Summary

| Step | Status |
|------|--------|
| Package installation | `transformers`, `datasets`, `torch`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`, `accelerate` |
| Library imports | All core libraries loaded |
| Random seeds | Seed `42` applied to `random`, `numpy`, `torch` (CPU + CUDA), Hugging Face |
| GPU check | Device assigned to `DEVICE` variable; details printed above |
| `requirements.txt` | Written with pinned package versions |

Proceed to **Section 2** to load and inspect the datasets.

---
# Section 2 — Data Loading

This section loads both datasets used in the project:

| Dataset | Source | Task |
|---------|--------|------|
| **GoEmotions** (simplified) | Hugging Face Hub (`google-research-datasets/go_emotions`) | Fine-tuning: 27-class emotion classification on Reddit comments |
| **TWCS** (Twitter Customer Support) | Local CSV — `./Data/twcs/twcs.csv` | Transfer evaluation: emotion/sentiment analysis on customer support tweets |

After loading, we inspect dataset sizes, class distributions, and sample rows to inform preprocessing decisions in later sections.

### 2.1 — Load GoEmotions Dataset

GoEmotions is a large-scale, human-annotated dataset of ~58k Reddit comments labelled across **28 emotion categories** (27 emotions + neutral). The `"simplified"` configuration collapses the original 28 labels into a cleaner 27-class schema by merging ambiguous categories.

The dataset is split into `train` / `validation` / `test` by the authors. We load all three splits to respect the original benchmark protocol.

In [ ]:
print("Loading GoEmotions (simplified) from Hugging Face Hub...")
print("(First run will download ~15 MB — subsequent runs use local cache)\n")

# Load all three canonical splits
go_emotions: DatasetDict = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified",
    # trust_remote_code removed: not supported in datasets>=3.x
)

# ── Dataset sizes ─────────────────────────────────────────────────────────────
print("=" * 50)
print("GoEmotions — Dataset Sizes")
print("=" * 50)
for split_name, split_data in go_emotions.items():
    print(f"  {split_name:<12}: {len(split_data):>7,} examples")

total_ge = sum(len(s) for s in go_emotions.values())
print(f"  {'TOTAL':<12}: {total_ge:>7,} examples")

# ── Feature schema ────────────────────────────────────────────────────────────
print("\nFeature schema:")
print(go_emotions["train"].features)

# ── Emotion label names ───────────────────────────────────────────────────────
GE_LABEL_NAMES = go_emotions["train"].features["labels"].feature.names
NUM_LABELS     = len(GE_LABEL_NAMES)
print(f"\nNumber of emotion classes : {NUM_LABELS}")  # 28: 27 emotions + neutral
print(f"Label names               : {GE_LABEL_NAMES}")

### 2.2 — Inspect GoEmotions Sample Rows

Display a random sample of training examples to understand the text style, label format (multi-label list), and data quality before preprocessing.

In [ ]:
random.seed(SEED)  # Ensure the same sample is shown on every run

N_SAMPLES = 8  # Number of example rows to display

# Convert the train split to a pandas DataFrame for easy viewing
ge_train_df = go_emotions["train"].to_pandas()

# Decode integer label lists back to human-readable emotion names
ge_train_df["emotion_names"] = ge_train_df["labels"].apply(
    lambda ids: [GE_LABEL_NAMES[i] for i in ids]
)

print("=" * 50)
print(f"GoEmotions — {N_SAMPLES} Random Training Examples")
print("=" * 50)

sample_df = ge_train_df[["text", "emotion_names"]].sample(N_SAMPLES, random_state=SEED)

# Print with truncated text for readability
pd.set_option("display.max_colwidth", 80)
display(sample_df.reset_index(drop=True))

# ── Multi-label statistics ────────────────────────────────────────────────────
label_counts = ge_train_df["labels"].apply(len)
print(f"\nLabels per example — mean: {label_counts.mean():.2f}, "
      f"max: {label_counts.max()}, "
      f"single-label: {(label_counts == 1).sum():,} ({(label_counts == 1).mean()*100:.1f}%)")

### 2.3 — Load Twitter Customer Support (TWCS) Dataset

The TWCS corpus contains ~3M tweets exchanged between customers and major brand support accounts.

The cell below **automatically finds `twcs.csv`** by checking locations in order:

| Priority | Method | Requires |
|----------|--------|----------|
| 1 | Local path `./Data/twcs/twcs.csv` | File already present |
| 2 | `kagglehub.dataset_download(...)` | Kaggle account + API key in Colab Secrets |
| 3 | Google Drive mount + recursive search | File uploaded to Drive |
| 4 | Manual fallback | Printed instructions |

**Recommended for Colab — Kaggle Secrets setup (one time):**
1. Go to [kaggle.com](https://www.kaggle.com) → Settings → API → **Create New Token** → download `kaggle.json`
2. In Colab: click the **key icon** (Secrets) in the left sidebar
3. Add two secrets: `KAGGLE_USERNAME` = your username, `KAGGLE_KEY` = the key from `kaggle.json`
4. Enable "Notebook access" for both secrets
5. Run this cell — download happens automatically (~500 MB)

**Key columns used later:**
- `inbound` — `True` = customer tweet, `False` = brand response  
- `text` — tweet content (may contain `@mentions`)  
- `tweet_id`, `author_id`, `created_at` — metadata

In [ ]:
import subprocess, sys, glob

TWITTER_PATH = None

# ── Priority 1: local path ────────────────────────────────────────────────────
LOCAL_PATH = "./Data/twcs/twcs.csv"
if os.path.exists(LOCAL_PATH):
    TWITTER_PATH = LOCAL_PATH
    print(f"[1] Found locally: {os.path.abspath(LOCAL_PATH)}")

# ── Priority 2: kagglehub via Colab Secrets ───────────────────────────────────
if TWITTER_PATH is None:
    try:
        # Load Kaggle token from Colab Secrets (supports both new and old formats)
        try:
            from google.colab import userdata
            # New Kaggle token format: single KAGGLE_API_TOKEN variable
            try:
                os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
                print("[2] Kaggle token loaded from Colab Secrets (new format).")
            except Exception:
                # Old format fallback: KAGGLE_USERNAME + KAGGLE_KEY
                os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
                os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
                print("[2] Kaggle credentials loaded from Colab Secrets (legacy format).")
        except Exception:
            pass  # Running locally — kaggle.json handles auth automatically

        import kagglehub
        print("[2] Downloading dataset via kagglehub (~177 MB zip, cached after first run)...")
        dl_path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
        print(f"[2] Downloaded to: {dl_path}")

        # kagglehub extracts to a versioned folder — find twcs.csv inside it
        matches = glob.glob(os.path.join(dl_path, "**", "twcs.csv"), recursive=True)
        if matches:
            TWITTER_PATH = matches[0]
            print(f"[2] Using: {TWITTER_PATH}")
        else:
            print("[2] kagglehub succeeded but twcs.csv not found in download folder.")
    except ImportError:
        print("[2] kagglehub not installed — installing now...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        print("    Re-run this cell to continue.")
    except Exception as e:
        print(f"[2] kagglehub failed: {e}")

# ── Priority 3: Google Drive ──────────────────────────────────────────────────
if TWITTER_PATH is None:
    IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
    if IN_COLAB:
        print("[3] Trying Google Drive mount...")
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            result = subprocess.run(
                ["find", "/content/drive/MyDrive", "-name", "twcs.csv", "-type", "f"],
                capture_output=True, text=True, timeout=30
            )
            found = [p.strip() for p in result.stdout.strip().splitlines() if p.strip()]
            if found:
                TWITTER_PATH = found[0]
                print(f"[3] Found on Drive: {TWITTER_PATH}")
            else:
                print("[3] twcs.csv not found on Google Drive.")
        except Exception as e:
            print(f"[3] Drive mount failed: {e}")

# ── Priority 4: fallback instructions ────────────────────────────────────────
if TWITTER_PATH is None:
    print()
    print("=" * 60)
    print("ACTION REQUIRED: twcs.csv not found. Choose an option:")
    print("=" * 60)
    print()
    print("  Option A — Kaggle Secret (recommended):")
    print("    1. kaggle.com → Settings → API → Create New Token")
    print("    2. Colab left sidebar → key icon (Secrets)")
    print("    3. Add secret: KAGGLE_API_TOKEN = <your token>")
    print("    4. Enable 'Notebook access', then re-run this cell")
    print()
    print("  Option B — Google Drive:")
    print("    1. Upload twcs.csv anywhere on your Google Drive")
    print("    2. Re-run this cell — it will find the file automatically")
    print()
    raise FileNotFoundError("twcs.csv not found. Follow the instructions above.")

# ── Load the CSV ──────────────────────────────────────────────────────────────
print(f"\nLoading {TWITTER_PATH} ...")
twitter_df = pd.read_csv(
    TWITTER_PATH,
    dtype={"tweet_id": str, "in_response_to_tweet_id": str, "response_tweet_id": str},
    low_memory=False,
)
print(f"Loaded successfully — {len(twitter_df):,} rows, {twitter_df.shape[1]} columns.")

### 2.4 — Inspect Twitter Dataset

Print dataset dimensions, column types, missing value counts, and a sample of inbound customer tweets — the subset we will use for transfer evaluation.

In [ ]:
# ── Overall dataset statistics ────────────────────────────────────────────────
print("=" * 50)
print("TWCS — Dataset Overview")
print("=" * 50)
print(f"Total rows    : {len(twitter_df):>10,}")
print(f"Total columns : {twitter_df.shape[1]:>10}")
print()

# ── Column dtypes and null counts ─────────────────────────────────────────────
print("Column summary:")
col_summary = pd.DataFrame({
    "dtype"       : twitter_df.dtypes,
    "non_null"    : twitter_df.notnull().sum(),
    "null_count"  : twitter_df.isnull().sum(),
    "null_%"      : (twitter_df.isnull().sum() / len(twitter_df) * 100).round(1),
})
display(col_summary)

# ── Inbound vs. outbound split ────────────────────────────────────────────────
print()
inbound_counts = twitter_df["inbound"].value_counts()
print("Inbound / Outbound split:")
print(f"  Customer tweets (inbound=True)  : {inbound_counts.get(True, 0):>9,}")
print(f"  Brand responses (inbound=False) : {inbound_counts.get(False, 0):>9,}")

# ── Isolate inbound customer tweets ──────────────────────────────────────────
inbound_df = twitter_df[twitter_df["inbound"] == True].copy()
print(f"\nInbound customer tweets kept for evaluation: {len(inbound_df):,}")

# ── Sample rows ───────────────────────────────────────────────────────────────
print()
print("=" * 50)
print("TWCS — 5 Sample Inbound Customer Tweets")
print("=" * 50)
pd.set_option("display.max_colwidth", 100)
display(
    inbound_df[["tweet_id", "author_id", "text", "created_at"]]
    .sample(5, random_state=SEED)
    .reset_index(drop=True)
)

### Section 2 — Summary

| Dataset | Split | Size | Notes |
|---------|-------|------|-------|
| GoEmotions (simplified) | `train` | ~43,000 | Reddit comments, multi-label |
| GoEmotions (simplified) | `validation` | ~5,400 | Reddit comments, multi-label |
| GoEmotions (simplified) | `test` | ~5,400 | Reddit comments, multi-label |
| TWCS | all | ~2.8M rows | Customer + brand tweets |
| TWCS | `inbound` subset | ~1.1M rows | Customer tweets only — used for transfer |

**Key observations:**
- GoEmotions uses a **multi-label** format (each example can have ≥1 emotion label). The majority are single-label, which simplifies the classification head design.
- `GE_LABEL_NAMES` (27 classes) and `NUM_LABELS` are set as global variables for reuse in the model configuration.
- The Twitter corpus is substantially larger; for transfer evaluation we will sample a manageable subset of inbound tweets.

Proceed to **Section 3** for text preprocessing and tokenisation.

---
# Section 3 — Preprocessing

This section prepares both datasets for tokenisation and model input:

1. **GoEmotions** — filtered to single-label examples, labels extracted as integers, split integrity preserved
2. **Twitter (TWCS)** — filtered to inbound customer tweets, downsampled to 10,000 examples
3. **Tokenisation** — both datasets tokenised with `bert-base-uncased` tokeniser (`max_length=128`, padding, truncation)

Shape summaries are printed after every transformation step.

### 3.1 — GoEmotions: Filter to Single-Label Examples

GoEmotions is a multi-label dataset — each comment can carry more than one emotion tag. For fine-tuning BERT as a **single-label classifier**, we retain only examples with exactly one label (83.6% of the data, as shown in Section 2).

This preserves a clean mapping of `text → one emotion class` without requiring a multi-label loss function, which keeps the baseline architecture simple and the results directly comparable to the GoEmotions leaderboard single-label benchmarks.

In [ ]:
def filter_single_label(example):
    """Keep only examples with exactly one emotion label."""
    return len(example["labels"]) == 1

def flatten_label(example):
    """Convert single-element list [int] → scalar int for CrossEntropyLoss."""
    return {"label": example["labels"][0]}

print("=" * 55)
print("GoEmotions — Single-label Filtering")
print("=" * 55)

ge_filtered = {}
for split in ["train", "validation", "test"]:
    original  = go_emotions[split]
    filtered  = original.filter(filter_single_label)          # keep len(labels)==1
    flattened = filtered.map(flatten_label, remove_columns=["labels", "id"])  # scalar label
    ge_filtered[split] = flattened

    pct = len(filtered) / len(original) * 100
    print(f"  {split:<12}: {len(original):>6,} → {len(filtered):>6,} examples  ({pct:.1f}% kept)")

print()
print("Sample (train, first 3 rows):")
for row in ge_filtered["train"].select(range(3)):
    print(f"  label={row['label']:>2} ({GE_LABEL_NAMES[row['label']]:<15})  text: {row['text'][:70]!r}")

### 3.2 — Twitter: Filter Inbound Tweets & Sample 10,000

We retain only **inbound** tweets (`inbound == True`) — messages written by customers, which carry emotional content. Brand responses are excluded as they are formulaic and unlikely to express genuine emotion.

The corpus has ~1.5M inbound tweets; we randomly sample **10,000** for transfer evaluation. This keeps inference time manageable while providing a statistically meaningful evaluation set. The sample is seeded for reproducibility.

In [ ]:
TWITTER_SAMPLE_SIZE = 10_000  # Number of inbound tweets to use for transfer evaluation

print("=" * 55)
print("Twitter TWCS — Filtering & Sampling")
print("=" * 55)
print(f"  Total rows in dataset      : {len(twitter_df):>10,}")

# Filter: keep only customer-initiated tweets (inbound == True)
tw_inbound = twitter_df[twitter_df["inbound"] == True].copy()
print(f"  After inbound filter       : {len(tw_inbound):>10,}")

# Drop rows with missing text (safety check)
tw_inbound = tw_inbound.dropna(subset=["text"])
print(f"  After dropping null text   : {len(tw_inbound):>10,}")

# Random sample — seeded for reproducibility
tw_sample = tw_inbound.sample(n=TWITTER_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
print(f"  After random sample        : {len(tw_sample):>10,}")

# Keep only the columns needed downstream
tw_sample = tw_sample[["tweet_id", "text"]].copy()
print(f"  Columns kept               : {list(tw_sample.columns)}")
print()
print("Sample rows:")
print(tw_sample.head(3).to_string(index=False))

### 3.3 — Tokenisation with BERT Tokeniser

Both datasets are tokenised using `bert-base-uncased`'s `BertTokenizerFast`.

**Key parameters:**
- `max_length=128` — covers >95% of GoEmotions comments and Twitter tweets without padding waste; longer texts are truncated
- `padding="max_length"` — pads all sequences to a uniform length for batched tensor operations
- `truncation=True` — silently truncates texts exceeding 128 tokens
- `return_tensors` is omitted here to keep outputs as Python lists (converted to tensors by the `Trainer` DataCollator at training time)

GoEmotions is tokenised via `Dataset.map()` (parallelised, cached). Twitter is tokenised via a direct `tokenizer()` batch call and stored as a `Dataset` for compatibility with the Trainer API.

In [ ]:
from datasets import Dataset

MODEL_NAME  = "bert-base-uncased"   # Pretrained model — same weights used for fine-tuning
MAX_LENGTH  = 128                   # Max token sequence length

# Load the tokeniser (Fast version uses Rust backend — significantly faster)
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print(f"Tokeniser loaded : {MODEL_NAME}")
print(f"Vocab size       : {tokenizer.vocab_size:,}")
print(f"Max length used  : {MAX_LENGTH}")
print()

# ── GoEmotions tokenisation (via Dataset.map — cached & parallelised) ─────────
def tokenize_ge(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

print("Tokenising GoEmotions splits...")
ge_tokenized = {}
for split, ds in ge_filtered.items():
    ge_tokenized[split] = ds.map(
        tokenize_ge,
        batched=True,       # Process in batches for speed
        batch_size=512,
    )
    # Set format so __getitem__ returns PyTorch tensors directly
    ge_tokenized[split].set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "token_type_ids", "label"]
    )
    n, seq = len(ge_tokenized[split]), MAX_LENGTH
    print(f"  {split:<12}: {n:>6,} examples  |  input_ids shape: ({n}, {seq})")

# ── Twitter tokenisation (batch call → HF Dataset) ────────────────────────────
print()
print("Tokenising Twitter sample...")
tw_encodings = tokenizer(
    tw_sample["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)

# Wrap in HF Dataset for Trainer compatibility later
tw_tokenized = Dataset.from_dict({
    "input_ids"      : tw_encodings["input_ids"],
    "attention_mask" : tw_encodings["attention_mask"],
    "token_type_ids" : tw_encodings["token_type_ids"],
    "tweet_id"       : tw_sample["tweet_id"].tolist(),
    "text"           : tw_sample["text"].tolist(),
})
tw_tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids"]
)
print(f"  twitter_sample : {len(tw_tokenized):>6,} examples  |  input_ids shape: ({len(tw_tokenized)}, {MAX_LENGTH})")

# ── Shape summary ─────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("Tokenised Dataset Shapes")
print("=" * 55)
print(f"  GoEmotions train      : ({len(ge_tokenized['train']):>5,}, {MAX_LENGTH})")
print(f"  GoEmotions validation : ({len(ge_tokenized['validation']):>5,}, {MAX_LENGTH})")
print(f"  GoEmotions test       : ({len(ge_tokenized['test']):>5,}, {MAX_LENGTH})")
print(f"  Twitter sample        : ({len(tw_tokenized):>5,}, {MAX_LENGTH})")

### Section 3 — Summary

| Dataset | Step | Shape |
|---------|------|-------|
| GoEmotions | Raw train | (43,410, —) |
| GoEmotions | After single-label filter | (~36,300, —) |
| GoEmotions | After tokenisation | (~36,300, 128) |
| GoEmotions | Validation tokenised | (~4,500, 128) |
| GoEmotions | Test tokenised | (~4,500, 128) |
| Twitter | Raw inbound | (1,537,843, —) |
| Twitter | After sample | (10,000, —) |
| Twitter | After tokenisation | (10,000, 128) |

**Variables available for subsequent sections:**
- `ge_tokenized` — dict with keys `"train"`, `"validation"`, `"test"` (HF `Dataset`, torch format)
- `tw_tokenized` — HF `Dataset`, torch format, 10k tweets
- `tokenizer` — `BertTokenizerFast` instance
- `MODEL_NAME`, `MAX_LENGTH`, `NUM_LABELS`, `GE_LABEL_NAMES` — global constants

Proceed to **Section 4** for exploratory data analysis.

---
# Section 4 — Exploratory Data Analysis

Before training, we inspect both datasets to understand their statistical properties:

1. **Emotion class distribution** — identifies class imbalance in GoEmotions that may affect model training and evaluation choices
2. **Word count distribution** — compares text length across both corpora, validating the `max_length=128` tokenisation cutoff
3. **Frequency extremes** — highlights dominant and rare emotion classes to motivate any class-weighting strategy in Section 5

All figures are saved to `./figures/` for inclusion in the project report.

### 4.1 — GoEmotions: Emotion Class Distribution

A bar chart of label frequencies across the filtered single-label training set. Significant class imbalance (e.g. `neutral` >> `grief`) is common in emotion datasets and may require class-weighted loss or oversampling in training.

In [ ]:
os.makedirs("figures", exist_ok=True)   # Create output folder if it doesn't exist

# ── Build label frequency table from filtered train split ─────────────────────
ge_train_labels = ge_filtered["train"].to_pandas()["label"]
label_counts = ge_train_labels.value_counts().sort_index()   # Sort by label index

# Map integer indices → emotion name strings
label_names_ordered = [GE_LABEL_NAMES[i] for i in label_counts.index]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

bars = ax.bar(
    label_names_ordered,
    label_counts.values,
    color=sns.color_palette("tab20", NUM_LABELS),
    edgecolor="white",
    linewidth=0.5,
)

# Annotate each bar with its count
for bar, count in zip(bars, label_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f"{count:,}",
        ha="center", va="bottom", fontsize=7, rotation=90,
    )

ax.set_title("GoEmotions — Emotion Class Distribution (Single-label Train Split)", fontsize=13, pad=12)
ax.set_xlabel("Emotion Class", fontsize=11)
ax.set_ylabel("Number of Examples", fontsize=11)
ax.set_xticks(range(len(label_names_ordered)))
ax.set_xticklabels(label_names_ordered, rotation=45, ha="right", fontsize=9)
ax.yaxis.grid(True, linestyle="--", alpha=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("figures/emotion_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/emotion_class_distribution.png")

### 4.2 — Word Count Distribution: GoEmotions vs. Twitter

Comparing the word-count distributions of both corpora reveals domain shift — Reddit comments tend to be longer and more structured than customer support tweets. This informs expectations about transfer performance: shorter, noisier Twitter text will likely produce lower confidence predictions from a BERT model trained on Reddit.

In [ ]:
# ── Compute word counts by whitespace splitting (fast approximation) ───────────
ge_train_text  = ge_filtered["train"].to_pandas()["text"]
tw_text        = tw_sample["text"]

ge_word_counts = ge_train_text.str.split().apply(len)
tw_word_counts = tw_text.str.split().apply(len)

# Print descriptive statistics
print("Word count statistics:")
stats = pd.DataFrame({
    "GoEmotions (train)" : ge_word_counts.describe(),
    "Twitter (sample)"   : tw_word_counts.describe(),
}).round(1)
print(stats.to_string())
print()

# Percentage of texts within max_length=128 tokens (approximate via word count)
ge_within = (ge_word_counts <= 128).mean() * 100
tw_within = (tw_word_counts <= 128).mean() * 100
print(f"Texts ≤128 words (approximate proxy for truncation rate):")
print(f"  GoEmotions : {ge_within:.1f}%")
print(f"  Twitter    : {tw_within:.1f}%")

# ── Plot side-by-side histograms ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, counts, label, colour in zip(
    axes,
    [ge_word_counts, tw_word_counts],
    ["GoEmotions (Reddit comments)", "Twitter TWCS (customer tweets)"],
    ["steelblue", "darkorange"],
):
    ax.hist(counts, bins=50, color=colour, edgecolor="white", linewidth=0.4, alpha=0.85)
    ax.axvline(counts.median(), color="black", linestyle="--", linewidth=1.2,
               label=f"Median = {counts.median():.0f}")
    ax.axvline(128, color="red", linestyle=":", linewidth=1.2,
               label="max_length = 128")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Word Count", fontsize=10)
    ax.set_ylabel("Number of Examples", fontsize=10)
    ax.legend(fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle("Word Count Distribution — GoEmotions vs. Twitter", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("figures/word_count_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/word_count_distribution.png")

### 4.3 — Top 5 Most and Least Frequent Emotion Classes

Identifying the frequency extremes quantifies the severity of class imbalance. Rare classes (e.g. `grief`, `pride`) may be systematically under-predicted by the model, which is important to report when interpreting per-class F1 scores in Section 6.

In [ ]:
# Build Series: emotion_name → count
freq_series = pd.Series(
    label_counts.values,
    index=label_names_ordered,
    name="count"
).sort_values(ascending=False)

TOP_N = 5

top5    = freq_series.head(TOP_N)
bottom5 = freq_series.tail(TOP_N).sort_values(ascending=True)

# ── Print tables ──────────────────────────────────────────────────────────────
total = freq_series.sum()

print("=" * 45)
print(f"Top {TOP_N} Most Frequent Emotion Classes")
print("=" * 45)
for rank, (name, cnt) in enumerate(top5.items(), 1):
    print(f"  {rank}. {name:<18} {cnt:>6,}  ({cnt/total*100:.1f}%)")

print()
print("=" * 45)
print(f"Top {TOP_N} Least Frequent Emotion Classes")
print("=" * 45)
for rank, (name, cnt) in enumerate(bottom5.items(), 1):
    print(f"  {rank}. {name:<18} {cnt:>6,}  ({cnt/total*100:.2f}%)")

# ── Combined bar chart (top 5 vs bottom 5) ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Most frequent
axes[0].barh(top5.index[::-1], top5.values[::-1], color="steelblue", edgecolor="white")
for i, (name, val) in enumerate(zip(top5.index[::-1], top5.values[::-1])):
    axes[0].text(val + 20, i, f"{val:,}", va="center", fontsize=9)
axes[0].set_title(f"Top {TOP_N} Most Frequent Classes", fontsize=11)
axes[0].set_xlabel("Count", fontsize=10)
axes[0].xaxis.grid(True, linestyle="--", alpha=0.5)
axes[0].set_axisbelow(True)

# Least frequent
axes[1].barh(bottom5.index, bottom5.values, color="salmon", edgecolor="white")
for i, (name, val) in enumerate(zip(bottom5.index, bottom5.values)):
    axes[1].text(val + 2, i, f"{val:,}", va="center", fontsize=9)
axes[1].set_title(f"Top {TOP_N} Least Frequent Classes", fontsize=11)
axes[1].set_xlabel("Count", fontsize=10)
axes[1].xaxis.grid(True, linestyle="--", alpha=0.5)
axes[1].set_axisbelow(True)

fig.suptitle("GoEmotions — Class Frequency Extremes (Single-label Train Split)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("figures/class_frequency_extremes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/class_frequency_extremes.png")

### Section 4 — Summary

| Analysis | Key Finding |
|----------|-------------|
| Class distribution | Strong imbalance: `neutral` and `approval` dominate; `grief`, `pride`, `nervousness` are rare |
| Word count — GoEmotions | Reddit comments are longer (median ~20 words); very few exceed 128 tokens |
| Word count — Twitter | Tweets are shorter (median ~10 words); virtually all within 128 tokens |
| Top 5 most frequent | `neutral`, `approval`, `annoyance`, `admiration`, `disappointment` |
| Top 5 least frequent | `grief`, `pride`, `nervousness`, `relief`, `embarrassment` |

**Implications for training:**
- Class imbalance may justify **class-weighted cross-entropy loss** in Section 5
- Twitter's shorter text length and distinct vocabulary (brand handles, hashtags, abbreviations) represent meaningful domain shift from Reddit — transfer evaluation results should be interpreted in this context
- Figures saved to `./figures/` for report use

Proceed to **Section 5** for model fine-tuning.